# EWF + SQD Demo — Glycine (Simplest Amino Acid)

This notebook demonstrates three computational approaches for molecular quantum
calculations on **glycine** (H₂N–CH₂–COOH), the simplest amino acid. It
replicates the methodology from *"Molecular Quantum Computation on a Protein"*
at a scale that runs comfortably on a laptop — with one real QPU fragment.

| Method | Embedding | Solver(s) | QPU? |
|--------|-----------|-----------|------|
| 1 | None | Full-system CCSD | No |
| 2 | EWF | CCSD per fragment | No |
| 3 | EWF | FCI (small frags) + **SQD** (N-fragment) | **Yes** |

### Why glycine?
Glycine has 5 heavy atoms (N, Cα, C=O, O×2). IAO fragmentation with an MP2
bath at `truncation=1e-4` produces 10 fragments. The nitrogen fragment grows
to ~11 cluster orbitals — just above the FCI threshold — making it the
natural candidate for SQD. All other fragments stay ≤ 10 orbitals and run
as fast FCI in seconds.

### Fragment map (STO-3G, `orbital_threshold=11`)
```
Frag 0  N-fragment   ~11 orb  → SQD on ibm_pittsburgh  (22 qubits)
Frag 1  Cα-fragment  ~10 orb  → FCI
Frag 2  C-fragment    ~9 orb  → FCI
Frag 3  O-fragment    ~8 orb  → FCI
Frag 4  O-fragment    ~8 orb  → FCI
Frags 5–9  H ×5       1–3 orb → FCI
```

## 0 — Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyscf
import yaml
from pyscf import cc, scf

from quantum_fragment_methods.application.embedding import EWF
from quantum_fragment_methods.application.solvers.classical_zoo import CCSD, FCI
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
from quantum_fragment_methods.workflow import QFWorkflow

In [ ]:
# ---------------------------------------------------------------------------
# Paths — resolved relative to this notebook so the demo works whether the
# repo is mounted at /workspace or checked out locally.
# ---------------------------------------------------------------------------
DEMO_DIR = Path("__file__").resolve().parent
XYZ_FILE = DEMO_DIR / "system_glycine.xyz"
CONFIG_FILE = DEMO_DIR / "config_glycine_sto-3g.yaml"

# Read geometry (skip the 2-line XYZ header)
xyz_lines = XYZ_FILE.read_text().splitlines()
geometry_data = "\n".join(xyz_lines[2:])

print(f"Molecule : {xyz_lines[1]}")
print(f"Atoms    : {xyz_lines[0].strip()}")
print(f"Config   : {CONFIG_FILE}")

In [ ]:
# Load config — all tunable parameters live here
with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

qpu_config      = config["qpu"]
sqd_config      = config["sqd"]
embedder_config = config["embedder"]["ewf"]
sel_config      = config["solver_selection"]
orbital_threshold = sel_config["orbital_threshold"]

print(f"Basis            : {config['workflow']['basis']}")
print(f"Bath type        : {embedder_config['bath_type']}")
print(f"Truncation       : {embedder_config['truncation']}")
print(f"Orbital threshold: {orbital_threshold}  (FCI if n_orb < {orbital_threshold}, else SQD)")
print(f"QPU backend      : {qpu_config['backend_name']}")
print(f"Shots            : {qpu_config['sampler_options']['default_shots']}")

---
## Method 1 — Full-system CCSD (reference)

Single coupled-cluster calculation on the entire glycine molecule, no
fragmentation. Serves as the accuracy benchmark.

- 10 atoms, STO-3G → 28 basis functions
- Wall time: ~5 s on a laptop

In [ ]:
mol = pyscf.gto.Mole()
mol.atom   = geometry_data
mol.unit   = "Angstrom"
mol.basis  = config["workflow"]["basis"]
mol.verbose = 3
mol.build()

In [ ]:
mf = scf.RHF(mol)
mf.kernel()
hf_energy = mf.e_tot
print(f"HF energy: {hf_energy:.8f} Ha")

In [ ]:
mycc = cc.CCSD(mf)
mycc.kernel()
energy_full_ccsd = mycc.e_tot

print(f"\n{'='*55}")
print(f"Method 1: Full-system CCSD")
print(f"{'='*55}")
print(f"HF energy      : {hf_energy:.8f} Ha")
print(f"CCSD energy    : {energy_full_ccsd:.8f} Ha")
print(f"Correlation    : {(energy_full_ccsd - hf_energy)*1000:.4f} mHa")
print(f"{'='*55}")

---
## Method 2 — EWF + CCSD (fragmented, all classical)

The Embedded Wave Function (EWF) method partitions glycine into IAO fragments,
adds an MP2 bath to each, then solves each fragment independently with CCSD.
This is the classical baseline for the hybrid method below.

- Fragmentation: IAO (one fragment per heavy atom + H pseudo-fragments)
- Bath: MP2, `truncation=1e-4`
- Solver: CCSD for all fragments
- Wall time: ~30–60 s on a laptop

In [ ]:
ewf_ccsd = EWF(
    bath_type  = embedder_config["bath_type"],
    truncation = embedder_config["truncation"],
)

workflow_ccsd = QFWorkflow(
    geometry      = geometry_data,
    basis         = config["workflow"]["basis"],
    embedder      = ewf_ccsd,
    fragmentation = embedder_config["fragmentation"],
    save_path     = "results_ewf_ccsd/",
)

workflow_ccsd.add_solver_rule(
    solver_factory = lambda frag: CCSD(),
    condition      = None,
    priority       = 0,
)

In [ ]:
print("Running EWF+CCSD...")
results_ewf_ccsd = workflow_ccsd.run()
energy_ewf_ccsd  = results_ewf_ccsd.total_energy

print(f"\n{'='*55}")
print(f"Method 2: EWF + CCSD")
print(f"{'='*55}")
print(f"Fragments      : {len(results_ewf_ccsd.fragment_energies)}")
print(f"Total energy   : {energy_ewf_ccsd:.8f} Ha")
print(f"Δ from CCSD ref: {(energy_ewf_ccsd - energy_full_ccsd)*1000:.4f} mHa")
print(f"{'='*55}")

---
## Method 3 — EWF + FCI/SQD (adaptive, hybrid quantum)

Same EWF fragmentation, but now with **adaptive solver selection**:

| Rule | Condition | Solver |
|------|-----------|--------|
| Priority 10 | `n_orbitals < 11` | FCI (exact, fast) |
| Priority 0  | `n_orbitals ≥ 11` | SQD on **ibm_pittsburgh** |

The nitrogen fragment (~11 orbitals) is the only one that exceeds the
threshold. Its 22-qubit LUCJ circuit is submitted to `ibm_pittsburgh`.

**QPU workflow (checkpoint-aware):**
1. Running this cell submits the QPU job and prints a job ID.
2. The job ID is saved to `results_ewf_adaptive/fragment_0/job_id.txt`.
3. Re-running after the job completes automatically resumes — no resubmission.
4. To force a fresh QPU submission: delete `job_id.txt` and `counts.npy`
   from that directory.

### 3a — Credentials

Fill in `config_glycine_sto-3g.yaml` with your IBM Quantum token and CRN,
**or** override inline below. Your credentials are never stored in the
notebook — the config file is gitignored.

```yaml
qpu:
  credentials:
    channel: ibm_cloud
    token:    "<YOUR_TOKEN_HERE>"
    instance: "<YOUR_CRN_HERE>"
```

In [ ]:
from quantum_fragment_methods.qpu import IBMQuantumBackend

# Initialize backend — reads token/instance from config_glycine_sto-3g.yaml
backend = IBMQuantumBackend(qpu_config)
backend.initialize()
backend.get_backend()

props = backend.get_backend_properties()
print(f"Backend        : {props['backend_name']}")
print(f"Qubits         : {props['num_qubits']}")

### 3b — Build workflow and solve

In [ ]:
ewf_adaptive = EWF(
    bath_type  = embedder_config["bath_type"],
    truncation = embedder_config["truncation"],
)

workflow_adaptive = QFWorkflow(
    geometry      = geometry_data,
    basis         = config["workflow"]["basis"],
    embedder      = ewf_adaptive,
    fragmentation = embedder_config["fragmentation"],
    save_path     = "results_ewf_adaptive/",
)

# Rule 1: FCI for small fragments (n_orb < orbital_threshold)
workflow_adaptive.add_solver_rule(
    solver_factory = lambda frag: FCI(),
    condition      = lambda frag: frag.n_orbitals < orbital_threshold,
    priority       = 10,
)

# Rule 2: SQD for the N-fragment (n_orb >= orbital_threshold)
# workflow_path is set per-fragment inside solve() so each fragment
# keeps its own job_id.txt / counts.npy checkpoint.
workflow_adaptive.add_solver_rule(
    solver_factory = lambda frag: SQDSolver(backend, config=sqd_config),
    condition      = None,
    priority       = 0,
)

In [ ]:
# Print solver assignment before running
print("Solver assignment (fragment sizes determined after EWF kernel runs):")
print(f"  n_orb < {orbital_threshold} → FCI")
print(f"  n_orb ≥ {orbital_threshold} → SQD  (ibm_pittsburgh)")
print()
print("SQD parameters:")
print(f"  shots/fragment   : {qpu_config['sampler_options']['default_shots']}")
print(f"  iterations       : {sqd_config['iterations']}")
print(f"  n_batches        : {sqd_config['n_batches']}")
print(f"  samples_per_batch: {sqd_config['samples_per_batch']}")
print(f"  classical_backend: {sqd_config['classical_backend']}")

In [ ]:
# Run. The QPU job is submitted synchronously; `--wait` polls until complete.
# If the job is still queued when the kernel times out, just re-run this
# cell — it will resume from the saved checkpoint automatically.
print("Running EWF + FCI/SQD ...")
print("=" * 55)
results_ewf_adaptive = workflow_adaptive.run()
energy_ewf_adaptive  = results_ewf_adaptive.total_energy

In [ ]:
fci_count = sum(1 for frag in results_ewf_adaptive.fragments
                if frag.n_orbitals < orbital_threshold)
sqd_count = len(results_ewf_adaptive.fragments) - fci_count

print(f"\n{'='*55}")
print(f"Method 3: EWF + FCI/SQD — COMPLETED")
print(f"{'='*55}")
print(f"Fragments        : {len(results_ewf_adaptive.fragment_energies)}")
print(f"  FCI fragments  : {fci_count}")
print(f"  SQD fragments  : {sqd_count}  (ibm_pittsburgh)")
print(f"Total energy     : {energy_ewf_adaptive:.8f} Ha")
print(f"Δ from CCSD ref  : {(energy_ewf_adaptive - energy_full_ccsd)*1000:.4f} mHa")
print(f"{'='*55}")

---
## Comparison and visualisation

In [ ]:
methods  = ["Full CCSD\n(no fragmentation)", "EWF+CCSD\n(fragmented)", "EWF+(FCI+SQD)\n(hybrid quantum)"]
energies = [energy_full_ccsd, energy_ewf_ccsd, energy_ewf_adaptive]
colors   = ["#2E86AB", "#A23B72", "#F18F01"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# — absolute energies —
bars = ax1.bar(methods, energies, color=colors, alpha=0.75, edgecolor="black", linewidth=1.2)
for bar, e in zip(bars, energies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{e:.6f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.set_ylabel("Total energy (Ha)", fontsize=11)
ax1.set_title("Total energy comparison", fontsize=12, fontweight="bold")
ax1.grid(axis="y", alpha=0.3, linestyle="--")

# — deviation from full-CCSD reference —
diffs = [0.0,
         (energy_ewf_ccsd     - energy_full_ccsd) * 1000,
         (energy_ewf_adaptive - energy_full_ccsd) * 1000]

bars2 = ax2.bar(methods, diffs, color=colors, alpha=0.75, edgecolor="black", linewidth=1.2)
for bar, d in zip(bars2, diffs):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() if d >= 0 else bar.get_height(),
             f"{d:.4f}",
             ha="center", va="bottom" if d >= 0 else "top",
             fontsize=9, fontweight="bold")
ax2.axhline(y=0, color="red", linestyle="--", linewidth=1.8, label="CCSD reference")
ax2.set_ylabel("Δ from full CCSD (mHa)", fontsize=11)
ax2.set_title("Deviation from full-CCSD reference", fontsize=12, fontweight="bold")
ax2.grid(axis="y", alpha=0.3, linestyle="--")
ax2.legend()

plt.suptitle("Glycine / STO-3G — EWF+SQD demo", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("energy_comparison_glycine.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: energy_comparison_glycine.png")

## Summary table

In [ ]:
import pandas as pd

n_frags = len(results_ewf_ccsd.fragment_energies)

summary = pd.DataFrame({
    "Method": ["Full CCSD", "EWF+CCSD", "EWF+(FCI+SQD)"],
    "Embedding": ["None", "EWF", "EWF"],
    "Solver(s)": ["CCSD", "CCSD × all frags", f"FCI × {fci_count} + SQD × {sqd_count}"],
    "Fragments": [1, n_frags, n_frags],
    "QPU": ["No", "No", "ibm_pittsburgh"],
    "Total energy (Ha)": [
        f"{energy_full_ccsd:.8f}",
        f"{energy_ewf_ccsd:.8f}",
        f"{energy_ewf_adaptive:.8f}",
    ],
    "Δ from ref (mHa)": [
        "0.0000",
        f"{(energy_ewf_ccsd - energy_full_ccsd)*1000:.4f}",
        f"{(energy_ewf_adaptive - energy_full_ccsd)*1000:.4f}",
    ],
})

print("\n" + "="*95)
print("SUMMARY — Glycine / STO-3G")
print("="*95)
print(summary.to_string(index=False))
print("="*95)

summary.to_csv("method_comparison_glycine.csv", index=False)
print("\nSaved: method_comparison_glycine.csv")

---
## Conclusion

This notebook demonstrated three levels of the EWF+SQD methodology on
glycine — the simplest amino acid — serving as a laptop-scale preview of the
full alanine calculation in the HPC demo
(`examples/hpc_demos/alanine_ewf_sqd_demo/`).

Key takeaways:

1. **Full-system CCSD** gives the reference energy but scales as O(N⁶) — infeasible for proteins.
2. **EWF+CCSD** recovers near-identical correlation energy by solving small embedded fragments independently. This is the classical baseline.
3. **EWF+(FCI+SQD)** replaces the most strongly-correlated fragment (N) with a quantum sampler. SQD should recover an energy *below* the CCSD reference for that fragment, as it is variational.

To scale up:
- Swap `system_glycine.xyz` → `system.xyz` (alanine) and use `config_alanine_sto-3g.yaml`
- Run the 4-step HPC workflow in `examples/hpc_demos/alanine_ewf_sqd_demo/`